# Notebook 02: Construcción del Dataset de Entrenamiento

**Entrada:** `data/processed/protein_labels.csv` (generado en el Notebook 01)  
**Salida:** `data/processed/dataset.csv` (proteínas con features fisicoquímicas y label)

## Estrategia de descarga de secuencias

La mayoría de los IDs provienen de NCBI (GenBank, RefSeq, GI numbers).
Usamos la **API NCBI Entrez** como fuente principal.
Los IDs de UniProt se recuperan directamente desde la API REST de UniProt.
Los IDs irrecuperables (PDB, IEDB internos, genbank_other) se descartan.

| Tipo | Cantidad | Fuente |
|---|---|---|
| `genbank_protein` | ~1,043 | NCBI Entrez |
| `gi_number` | ~57 | NCBI Entrez |
| `refseq` | ~48 | NCBI Entrez |
| `uniprot` + `uniprot_style` | ~164 | UniProt REST |
| `genbank_other` + `pdb` + `iedb` | ~53 | Descartar |

## 0. Instalación y configuración

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import re
from io import StringIO
from pathlib import Path
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

PROCESSED_DIR = Path('../data/processed')
print("Librerías cargadas correctamente.")

## 1. Cargar protein_labels.csv y clasificar IDs

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'protein_labels.csv')
print(f"Proteínas cargadas: {len(df):,}")
print(f"Distribución de labels: {df['label'].value_counts().to_dict()}")

df['protein_id'] = df['source_molecule_iri'].str.split('/').str[-1]

def classify_id(row):
    iri = str(row['source_molecule_iri'])
    pid = str(row['protein_id'])
    if 'uniprot.org' in iri:
        return 'uniprot'
    if 'ontology.iedb.org' in iri:
        return 'iedb'
    if 'ncbi.nlm.nih.gov' in iri:
        if re.match(r'^\d+$', pid):
            return 'gi_number'
        elif re.match(r'^[A-Z]{1,2}\d{5}(\.\d+)?$', pid):
            return 'uniprot_style'
        elif re.match(r'^[A-Z]{2}_?\d+(\.\d+)?$', pid):
            return 'refseq'
        elif re.match(r'^[A-Z]{3}\d{5}(\.\d+)?$', pid):
            return 'genbank_protein'
        elif re.match(r'^[A-Z0-9]{4}_[A-Z]$', pid):
            return 'pdb'
        else:
            return 'genbank_other'
    return 'unknown'

df['id_type'] = df.apply(classify_id, axis=1)
print(f"\nClasificación de IDs:")
print(df['id_type'].value_counts())

## 2. Funciones de descarga de secuencias

In [ ]:
def fetch_ncbi_fasta(accession, retries=3):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    params = {'db': 'protein', 'id': accession, 'rettype': 'fasta', 'retmode': 'text'}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.ok and r.text.strip().startswith('>'):
                record = next(SeqIO.parse(StringIO(r.text), 'fasta'))
                seq = str(record.seq)
                if len(seq) > 0:
                    return seq
        except Exception:
            time.sleep(2 ** attempt)
    return None


def fetch_uniprot_fasta(uniprot_id, retries=3):
    uid = uniprot_id.split('.')[0]
    url = f'https://rest.uniprot.org/uniprotkb/{uid}.fasta'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=20)
            if r.ok and r.text.strip().startswith('>'):
                record = next(SeqIO.parse(StringIO(r.text), 'fasta'))
                seq = str(record.seq)
                if len(seq) > 0:
                    return seq
        except Exception:
            time.sleep(2 ** attempt)
    return None


def fetch_batch_ncbi(ids, delay=0.4):
    results = {}
    batch_size = 200
    batches = [ids[i:i+batch_size] for i in range(0, len(ids), batch_size)]

    for i, batch in enumerate(batches):
        print(f"  Lote {i+1}/{len(batches)} ({len(batch)} IDs)...", end=' ')
        url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
        params = {'db': 'protein', 'id': ','.join(batch), 'rettype': 'fasta', 'retmode': 'text'}
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.ok and r.text.strip().startswith('>'):
                records = list(SeqIO.parse(StringIO(r.text), 'fasta'))
                fetched = 0
                for record in records:
                    seq = str(record.seq)
                    if len(seq) > 0:
                        results[record.id] = seq
                        fetched += 1
                print(f"OK → {fetched} secuencias")
            else:
                print(f"sin resultados (HTTP {r.status_code})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(delay)

    return results


print("Funciones definidas.")

## 3. Descargar secuencias

### 3.1 UniProt directo

In [ ]:
sequences = {}  # protein_id → secuencia

uniprot_ids = df[df['id_type'].isin(['uniprot', 'uniprot_style'])]['protein_id'].tolist()
print(f"[1/4] UniProt directo: {len(uniprot_ids)} proteínas")

for pid in uniprot_ids:
    seq = fetch_uniprot_fasta(pid)
    if seq:
        sequences[pid] = seq
    time.sleep(0.2)

recovered = sum(1 for p in uniprot_ids if p in sequences)
print(f"  Recuperadas: {recovered}/{len(uniprot_ids)}")

### 3.2 NCBI — GenBank, GI numbers y RefSeq

Descarga en lotes de 200 IDs. Puede tardar 10-15 minutos.

In [ ]:
ncbi_types = ['genbank_protein', 'gi_number', 'refseq']
ncbi_df    = df[df['id_type'].isin(ncbi_types)].copy()
ncbi_ids   = ncbi_df['protein_id'].tolist()

print(f"[2/4] NCBI Entrez (GenBank + GI + RefSeq): {len(ncbi_ids)} proteínas")
print(f"  Por tipo: {ncbi_df['id_type'].value_counts().to_dict()}")
print()

ncbi_results = fetch_batch_ncbi(ncbi_ids, delay=0.4)
print(f"\n  Total secuencias en caché NCBI: {len(ncbi_results)}")

In [ ]:
ncbi_index = {}
for ncbi_acc, seq in ncbi_results.items():
    ncbi_index[ncbi_acc] = seq
    ncbi_index[ncbi_acc.split('.')[0]] = seq

matched = 0
for pid in ncbi_ids:
    if pid not in sequences:
        seq = ncbi_index.get(pid) or ncbi_index.get(pid.split('.')[0])
        if seq:
            sequences[pid] = seq
            matched += 1

print(f"  Cruzados con protein_ids originales: {matched}")
print(f"  Total acumulado en sequences: {len(sequences)}")

In [ ]:
ncbi_missing = [pid for pid in ncbi_ids if pid not in sequences]
print(f"IDs NCBI sin secuencia tras cruce: {len(ncbi_missing)}")

if len(ncbi_missing) > 0:
    print("  Intentando descarga individual...")
    recovered_individual = 0
    for pid in ncbi_missing:
        seq = fetch_ncbi_fasta(pid)
        if seq:
            sequences[pid] = seq
            recovered_individual += 1
        time.sleep(0.4)
    print(f"  Recuperados individualmente: {recovered_individual}/{len(ncbi_missing)}")

print(f"\nTotal secuencias recuperadas: {len(sequences)}")

In [ ]:
df['sequence'] = df['protein_id'].apply(lambda pid: sequences.get(str(pid), None))

total       = len(df)
recuperadas = df['sequence'].notna().sum()
perdidas    = df['sequence'].isna().sum()

print(f"{'='*45}")
print(f"COBERTURA DE SECUENCIAS")
print(f"{'='*45}")
print(f"  Total proteínas:      {total:>6,}")
print(f"  Con secuencia:        {recuperadas:>6,} ({recuperadas/total:.1%})")
print(f"  Sin secuencia:        {perdidas:>6,} ({perdidas/total:.1%})")
print(f"\nSin secuencia por tipo de ID:")
print(df[df['sequence'].isna()]['id_type'].value_counts())
print(f"\nDistribución de labels en proteínas CON secuencia:")
print(df[df['sequence'].notna()]['label'].value_counts())

## 4. Calcular features fisicoquímicas con Biopython

Para cada proteína calculamos:
- **Longitud**: número de aminoácidos
- **Peso molecular**: en Daltons
- **Punto isoeléctrico**: pH al que la carga neta es cero
- **GRAVY**: índice de hidrofobicidad media
- **Composición de aminoácidos**: porcentaje de cada uno de los 20 aa estándar

In [ ]:
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def compute_features(seq):
    seq = re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', seq.upper())
    if len(seq) < 10:
        return None
    try:
        analysis = ProteinAnalysis(seq)
        features = {
            'length':            len(seq),
            'molecular_weight':  analysis.molecular_weight(),
            'isoelectric_point': analysis.isoelectric_point(),
            'gravy':             analysis.gravy(),
        }
        aa_comp = analysis.amino_acids_percent
        for aa in AMINO_ACIDS:
            features[f'aa_{aa}'] = aa_comp.get(aa, 0.0)
        return features
    except Exception:
        return None


print("Calculando features fisicoquímicas...")
feature_rows = []
errors = 0

for _, row in df[df['sequence'].notna()].iterrows():
    feats = compute_features(row['sequence'])
    if feats is not None:
        feats['protein_id']          = row['protein_id']
        feats['source_molecule']     = row['source_molecule']
        feats['source_molecule_iri'] = row['source_molecule_iri']
        feats['pathogen']            = row['pathogen']
        feats['label']               = row['label']
        feature_rows.append(feats)
    else:
        errors += 1

dataset = pd.DataFrame(feature_rows)
print(f"Features calculadas: {len(dataset):,} proteínas")
print(f"Errores de cálculo:  {errors}")
print(f"Columnas totales:    {len(dataset.columns)}")

## 5. Inspección del dataset resultante

In [ ]:
import matplotlib.pyplot as plt

feature_cols = ['length', 'molecular_weight', 'isoelectric_point', 'gravy'] + \
               [f'aa_{aa}' for aa in AMINO_ACIDS]

print("Estadísticas descriptivas de las features numéricas:")
dataset[feature_cols].describe().T.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
main_features = ['length', 'molecular_weight', 'isoelectric_point', 'gravy']
titles = ['Longitud', 'Peso molecular (Da)', 'Punto isoeléctrico', 'GRAVY (hidrofobicidad)']

for ax, feat, title in zip(axes, main_features, titles):
    for label, color in [(0, '#d9534f'), (1, '#5cb85c')]:
        subset = dataset[dataset['label'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'label={label} (n={len(subset)})')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Distribución de features por clase', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("Distribución de labels en el dataset final:")
vc = dataset['label'].value_counts()
print(vc)
print(f"\nDesbalance (label=1 / label=0): {vc[1] / vc[0]:.2f}")
print(f"\nPor patógeno:")
print(dataset.groupby(['pathogen', 'label']).size().unstack(fill_value=0))

In [ ]:
nulls = dataset[feature_cols].isnull().sum()
if nulls.sum() == 0:
    print("Sin valores nulos en las features. Dataset listo.")
else:
    print("Valores nulos encontrados:")
    print(nulls[nulls > 0])

## 6. Guardar dataset.csv

In [ ]:
OUTPUT_PATH = PROCESSED_DIR / 'dataset.csv'

meta_cols    = ['protein_id', 'source_molecule', 'source_molecule_iri', 'pathogen']
cols_ordered = meta_cols + feature_cols + ['label']
dataset[cols_ordered].to_csv(OUTPUT_PATH, index=False)

print(f"Guardado: {OUTPUT_PATH}")
print(f"Dimensiones: {len(dataset):,} filas × {len(cols_ordered)} columnas")
print(f"  - Metadatos:  {len(meta_cols)}")
print(f"  - Features:   {len(feature_cols)} (4 fisicoquímicas + 20 composición aa)")
print(f"  - Label:      1")
dataset[cols_ordered].head(3)

## 7. Resumen para el Notebook 03

In [ ]:
n1 = dataset['label'].value_counts().get(1, 0)
n0 = dataset['label'].value_counts().get(0, 0)

print("=" * 50)
print("RESUMEN DEL DATASET FINAL")
print("=" * 50)
print(f"  Proteínas totales:        {len(dataset):>6,}")
print(f"  label=1 (antigénicas):    {n1:>6,}")
print(f"  label=0 (no antigénicas): {n0:>6,}")
print(f"  Desbalance (1/0):         {n1/n0:.2f}")
print(f"  Features por proteína:    {len(feature_cols):>6}")
print("=" * 50)
print(f"\n→ En NB03 usaremos class_weight='balanced' en Random Forest")
print(f"→ Archivo listo: {OUTPUT_PATH}")